# **1. Opportunity signals**

This notebook builds an exploratory recruitment layer for identifying players aged 16–23 whose playing-time sequence is unusual for their age and competitive context.

The signal measures **revealed opportunity or selection propensity**, not talent. Item response models and statistical process monitoring remain candidate model families only after an expected-minutes model has been validated prospectively.

## **2. Statistical design**

1. `played`, `in squad`, and `not in squad` define the observed selection-decision universe.
2. `injured`, `absent`, and `played for a different team` are not treated as zero opportunity.
3. Each league–season–round reference is calculated with players of **all ages**.
4. The 16–23 age restriction is applied only after this normalization.
5. A descriptive opportunity residual is calculated within league, season, and integer age.
6. EWMA and CUSUM accumulate persistence and restart at each league spell. Their limits are not yet calibrated, so they are signals rather than alarms.

## **3. Data preparation and signal construction**

Stable player labels are derived from roster history, while unavailable states are excluded from the opportunity denominator. Minutes are normalized against the complete league–round population before age-specific residuals and sequential persistence measures are calculated.

In [1]:
from pathlib import Path
from collections import defaultdict
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
ASSETS = ROOT / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)

LEAGUE_NAMES = {
    'ENG-1': 'Premier League', 'SPA-1': 'LaLiga', 'GER-1': 'Bundesliga',
    'ITA-1': 'Serie A', 'FRA-1': 'Ligue 1', 'POR-1': 'Liga Portugal',
    'BR-1': 'Brasileirão', 'NED-1': 'Eredivisie',
    'BEL-1': 'Jupiler Pro League', 'ARG-1': 'Liga Profesional',
    'COL-1': 'Primera A', 'PAR-1': 'Primera División — Paraguai',
    'URU-1': 'Primera División — Uruguai', 'KSA-1': 'Saudi Pro League',
    'MEX-1': 'Liga MX', 'USA-1': 'Major League Soccer',
}
SELECTION_STATES = {'played', 'in squad', 'not in squad'}
EXCLUDED_STATES = {'injured', 'absent', 'played for a different team'}
MIN_AGE, MAX_AGE = 16, 23
EWMA_ALPHA = 0.25
CUSUM_ALLOWANCE = 0.25
MIN_ROUNDS_FOR_RANKING = 8
NON_USE_MIN_STREAK = 3
NON_USE_MIN_POST_SIGNAL_ROUNDS = 3
USAGE_HISTORICAL_MIN_ROUNDS = 8
USAGE_RECENT_ROUNDS = 5
USAGE_RECENCY_DAYS = 60

In [2]:
matches = pd.read_parquet(PROCESSED / 'player_match_minutes.parquet')
rosters = pd.read_parquet(RAW / 'roster_memberships.parquet')
teams = pd.read_parquet(RAW / 'teams.parquet')

state_audit = (matches.groupby('participation_state', dropna=False)
    .agg(rows=('game_id', 'size'), positive_minutes=('played', 'sum'),
         mean_minutes=('minutes_played', 'mean'))
    .sort_values('rows', ascending=False))
display(state_audit)

,rows,positive_minutes,mean_minutes
participation_state,,,
played,1146547,1146487,64.93295
not in squad,484445,0,0.00000
in squad,402788,0,0.00000
injured,174988,0,0.00000
played for a different team,104419,0,0.00000
absent,25075,0,0.00000


In [3]:
def pipe_unique(values):
    return '|'.join(sorted({str(value) for value in values.dropna()}))

def modal_value(values):
    modes = values.dropna().mode()
    return modes.iloc[0] if len(modes) else None

# Stable display identity. Position is the modal value in roster history.
identities = (rosters.sort_values('source_season_id')
    .groupby('player_id', as_index=False)
    .agg(player_name=('player_name', 'last'), player_position=('player_position', modal_value)))

team_lookup = (teams.rename(columns={
        'competition_code': 'competition_id', 'season_id': 'season_id',
        'team_id': 'club_id', 'team_name': 'club_name',
    })[['competition_id', 'season_id', 'club_id', 'club_name']]
    .assign(competition_id=lambda x: x.competition_id.astype(str),
            season_id=lambda x: x.season_id.astype('Int64'),
            club_id=lambda x: x.club_id.astype(str))
    .drop_duplicates(['competition_id', 'season_id', 'club_id']))

# Retain observed decisions; unavailability and another-club states do not become zero.
selection = matches.loc[matches['participation_state'].isin(SELECTION_STATES)].copy()
selection['club_id'] = selection['club_id'].astype(str)
selection['match_duration'] = pd.to_numeric(selection['match_duration'], errors='coerce')
selection['minutes_played'] = pd.to_numeric(selection['minutes_played'], errors='coerce').fillna(0)
selection['match_timestamp_utc'] = pd.to_datetime(selection['match_timestamp_utc'], utc=True)

ROUND_KEYS = ['competition_id', 'source_competition_id', 'season_id', 'round_id']
PLAYER_ROUND_KEYS = ['player_id', *ROUND_KEYS]
rounds = (selection.sort_values(['player_id', 'match_timestamp_utc', 'game_id'])
    .groupby(PLAYER_ROUND_KEYS, as_index=False, observed=True)
    .agg(match_date=('match_timestamp_utc', 'min'),
         club_id=('club_id', 'last'), club_ids=('club_id', pipe_unique),
         n_match_records=('game_id', 'nunique'),
         minutes_played=('minutes_played', 'sum'),
         available_minutes=('match_duration', 'sum'),
         age_decimal=('age_decimal', 'first'), age_years=('age_years', 'first'))
)
rounds['minutes_share'] = np.where(
    rounds['available_minutes'].gt(0), rounds['minutes_played'] / rounds['available_minutes'], np.nan
)
rounds['played'] = rounds['minutes_played'].gt(0)

# League-round reference: all ages before the 16–23 filter.
reference = (rounds.groupby(ROUND_KEYS, as_index=False, observed=True)
    .agg(n_players_observed=('player_id', 'nunique'),
         mean_minutes_all=('minutes_played', 'mean'),
         std_minutes_all=('minutes_played', lambda x: x.std(ddof=0)),
         share_played_all=('played', 'mean'))
)
rounds = rounds.merge(reference, on=ROUND_KEYS, how='left', validate='many_to_one')
rounds['minutes_vs_round_mean_all'] = np.where(
    rounds['mean_minutes_all'].gt(0), rounds['minutes_played'] / rounds['mean_minutes_all'], np.nan
)
rounds['round_percentile_all'] = rounds.groupby(ROUND_KEYS, observed=True)['minutes_played'].rank(
    method='average', pct=True
)

target = rounds.loc[rounds['age_years'].between(MIN_AGE, MAX_AGE, inclusive='both')].copy()
target = target.sort_values(['player_id', 'match_date', 'competition_id', 'round_id']).reset_index(drop=True)
previous_league = target.groupby('player_id', observed=True)['competition_id'].shift()
new_spell = previous_league.isna() | target['competition_id'].ne(previous_league)
target['league_spell_number'] = new_spell.astype(int).groupby(target['player_id']).cumsum()

# Resposta ordinal candidata para uma futura TRI de resposta gradual/crédito parcial.
target['response_category'] = np.select(
    [target['minutes_played'].eq(0), target['minutes_share'].le(1/3), target['minutes_share'].lt(.75)],
    [0, 1, 2], default=3
).astype('int8')

# Descriptive league-season-age benchmark; not a prospective score without backtesting.
PEER_KEYS = ['competition_id', 'source_competition_id', 'season_id', 'age_years']
peer = (target.groupby(PEER_KEYS, as_index=False, observed=True)
    .agg(peer_mean_percentile=('round_percentile_all', 'mean'),
         peer_sd_percentile=('round_percentile_all', lambda x: x.std(ddof=0)),
         peer_players=('player_id', 'nunique'))
)
target = target.merge(peer, on=PEER_KEYS, how='left', validate='many_to_one')
target['peer_residual_z'] = np.where(
    target['peer_sd_percentile'].gt(0),
    (target['round_percentile_all'] - target['peer_mean_percentile']) / target['peer_sd_percentile'],
    0.0
).clip(-4, 4)

def sequential_signals(group):
    player_id, spell = group.name
    group = group.sort_values(['match_date', 'round_id']).copy()
    group['player_id'] = player_id
    group['league_spell_number'] = spell
    group['spell_observation_number'] = np.arange(1, len(group) + 1)
    residual = group['peer_residual_z'].fillna(0.0)
    group['ewma_z'] = residual.ewm(alpha=EWMA_ALPHA, adjust=False).mean()
    group['ewma_signal_z'] = group['ewma_z'].where(group['spell_observation_number'].ge(5))
    cusum, running = [], 0.0
    for value in residual:
        running = max(0.0, running + float(value) - CUSUM_ALLOWANCE)
        cusum.append(running)
    group['cusum_positive'] = cusum
    group['rolling_minutes_5'] = group['minutes_played'].rolling(5, min_periods=1).sum()
    return group

target = (target.groupby(['player_id', 'league_spell_number'], group_keys=False, observed=True)
    .apply(sequential_signals, include_groups=False)
    .reset_index(drop=True))
target = target.merge(identities, on='player_id', how='left', validate='many_to_one')
target = target.merge(team_lookup, on=['competition_id', 'season_id', 'club_id'],
                      how='left', validate='many_to_one')
target['league_name'] = target['competition_id'].map(LEAGUE_NAMES).fillna(target['competition_id'])
target['club_name'] = target['club_name'].fillna(target['club_id'])

In [4]:
# One row per player for ranking and navigation.
grouped = target.groupby('player_id', observed=True)
summary = grouped.agg(
    player_name=('player_name', 'last'), player_position=('player_position', modal_value),
    first_date=('match_date', 'min'), last_date=('match_date', 'max'),
    first_age=('age_decimal', 'min'), latest_age=('age_decimal', 'last'),
    rounds=('round_id', 'size'), played_rounds=('played', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    mean_round_percentile=('round_percentile_all', 'mean'),
    peak_ewma_z=('ewma_signal_z', 'max'), latest_ewma_z=('ewma_z', 'last'),
    max_cusum=('cusum_positive', 'max'), league_spells=('league_spell_number', 'max'),
    leagues=('competition_id', 'nunique')
).reset_index()

latest = (target.sort_values('match_date').groupby('player_id', as_index=False).tail(1)
    [['player_id', 'competition_id', 'league_name', 'club_id', 'club_name']])
peak_source = target.dropna(subset=['ewma_signal_z'])
peak_index = peak_source.groupby('player_id', observed=True)['ewma_signal_z'].idxmax()
peak = (target.loc[peak_index, ['player_id', 'match_date', 'age_decimal', 'competition_id',
                                'league_name', 'club_name', 'ewma_z']]
    .rename(columns={'match_date': 'signal_date', 'age_decimal': 'signal_age',
                     'competition_id': 'signal_competition_id',
                     'league_name': 'signal_league_name', 'club_name': 'signal_club_name'}))
summary = summary.merge(latest, on='player_id', how='left').merge(peak, on='player_id', how='left')
summary['play_rate'] = summary['played_rounds'] / summary['rounds']
summary['rank_eligible'] = summary['rounds'].ge(MIN_ROUNDS_FOR_RANKING) & summary['peak_ewma_z'].notna()
summary['signal_percentile'] = np.nan
mask = summary['rank_eligible']
summary.loc[mask, 'signal_percentile'] = summary.loc[mask, 'peak_ewma_z'].rank(pct=True) * 100

# Monitoring layer: historical versus recent use across the full player trajectory.
# Transfers do not reset the sequence; unavailable states were already excluded.
data_cutoff = target['match_date'].max()
signal_date_lookup = summary.set_index('player_id')['signal_date'].to_dict()
usage_rows = []
for player_id, group in target.sort_values('match_date').groupby('player_id', observed=True):
    signal_date = signal_date_lookup.get(player_id)
    post_signal = (group.loc[group['match_date'].gt(signal_date)]
                   if pd.notna(signal_date) else group.iloc[0:0])
    unused_streak = 0
    for minutes in post_signal['minutes_played'].iloc[::-1]:
        if minutes > 0:
            break
        unused_streak += 1
    used = group.loc[group['minutes_played'].gt(0), 'match_date']
    recent = group.tail(USAGE_RECENT_ROUNDS)
    historical = group.iloc[:-USAGE_RECENT_ROUNDS]
    historical_available = historical['available_minutes'].sum()
    recent_available = recent['available_minutes'].sum()
    historical_use = (historical['minutes_played'].sum() / historical_available
                      if len(historical) >= USAGE_HISTORICAL_MIN_ROUNDS
                      and historical_available > 0 else np.nan)
    recent_use = (recent['minutes_played'].sum() / recent_available
                  if len(recent) == USAGE_RECENT_ROUNDS
                  and recent_available > 0 else np.nan)
    usage_rows.append({
        'player_id': player_id,
        'post_signal_rounds': len(post_signal),
        'unused_streak': unused_streak,
        'recent_5_minutes': recent['minutes_played'].sum(),
        'last_used_date': used.max() if len(used) else pd.NaT,
        'historical_usage_rounds': len(historical),
        'recent_usage_rounds': len(recent),
        'historical_usage_share': historical_use,
        'recent_usage_share': recent_use,
    })
usage = pd.DataFrame(usage_rows)
summary = summary.merge(usage, on='player_id', how='left', validate='one_to_one')
summary['last_observation_date'] = summary['last_date']
summary['days_since_observation'] = (
    data_cutoff - summary['last_observation_date']
).dt.total_seconds() / 86_400
summary['age_at_cutoff'] = summary['latest_age'] + (
    summary['days_since_observation'] / 365.2425
)
summary['usage_change_pp'] = 100 * (
    summary['recent_usage_share'] - summary['historical_usage_share']
)
summary['usage_plot_eligible'] = (
    summary['rank_eligible']
    & summary['age_at_cutoff'].ge(MIN_AGE)
    & summary['age_at_cutoff'].lt(MAX_AGE + 1)
    & summary['historical_usage_rounds'].ge(USAGE_HISTORICAL_MIN_ROUNDS)
    & summary['recent_usage_rounds'].eq(USAGE_RECENT_ROUNDS)
    & summary['historical_usage_share'].notna()
    & summary['recent_usage_share'].notna()
    & summary['days_since_observation'].le(USAGE_RECENCY_DAYS)
)
summary['non_use_flag'] = (
    summary['usage_plot_eligible']
    & summary['post_signal_rounds'].ge(NON_USE_MIN_POST_SIGNAL_ROUNDS)
    & summary['unused_streak'].ge(NON_USE_MIN_STREAK)
)
summary = summary.sort_values(['signal_percentile', 'total_minutes'], ascending=False).reset_index(drop=True)

audit = {
    'raw_match_rows': int(len(matches)),
    'selection_universe_rows': int(len(selection)),
    'excluded_unavailable_or_other_team_rows': int(len(matches) - len(selection)),
    'all_age_player_round_rows': int(len(rounds)),
    'target_player_round_rows': int(len(target)),
    'target_players': int(target['player_id'].nunique()),
    'players_in_multiple_leagues': int(summary['leagues'].gt(1).sum()),
    'rank_eligible_players': int(summary['rank_eligible'].sum()),
    'non_use_flagged_players': int(summary['non_use_flag'].sum()),
    'usage_plot_players': int(summary['usage_plot_eligible'].sum()),
    'league_rounds': int(len(reference)),
}
audit

{'raw_match_rows': 2338262,
 'selection_universe_rows': 2033780,
 'excluded_unavailable_or_other_team_rows': 304482,
 'all_age_player_round_rows': 2022931,
 'target_player_round_rows': 814065,
 'target_players': 18681,
 'players_in_multiple_leagues': 2465,
 'rank_eligible_players': 16016,
 'non_use_flagged_players': 965,
 'usage_plot_players': 2789,
 'league_rounds': 3994}

## **4. Methodological interpretation**

### **Item response modeling**

A future item response model should use the ordinal response `0 = did not play`, `1 = brief participation`, `2 = substantial participation`, and `3 = nearly or fully started`. The time-varying latent trait should be described as **selection propensity**. League, club, coach, position, and round may serve as difficulty facets or covariates, while players moving across leagues may serve as potential *anchor persons* for scale linking.

Local dependence remains central because players at the same club compete for finite lineup positions and minutes. A dynamic item response specification must therefore be compared with ordinal or hurdle mixed models and a roster-choice model. If invariance and fit fail, the item response interpretation should not be retained.

### **Statistical process monitoring**

EWMA and CUSUM should monitor residuals from an expected-minutes model rather than raw minutes. That model must account for age, league, position, club, squad strength, coach, availability, and schedule. Control limits require simulation or backtesting to manage false alarms and *average run length*. The prototype below describes persistence but does not yet claim statistical control.

## **5. Analytical outputs**

The exported tables preserve match-level decisions, league spells, sequential signals, and one player-level summary for ranking and navigation. Eligibility remains explicit so exploratory ordering is not mistaken for a validated scouting score.

In [5]:
# Analytical tables for subsequent stages.
target.to_parquet(PROCESSED / 'age_16_23_opportunity_signals.parquet', index=False)
summary.to_parquet(PROCESSED / 'player_opportunity_summary.parquet', index=False)

# Payload compacto para o index.html.
league_codes = sorted(LEAGUE_NAMES)
league_index = {code: index for index, code in enumerate(league_codes)}
club_names = sorted(target['club_name'].dropna().astype(str).unique())
club_index = {name: index for index, name in enumerate(club_names)}
histories = defaultdict(list)
for row in target.sort_values(['player_id', 'match_date']).itertuples(index=False):
    histories[str(row.player_id)].append([
        row.match_date.strftime('%Y-%m-%d'),
        round(float(row.age_decimal) * 1000),
        league_index[str(row.competition_id)],
        club_index[str(row.club_name)],
        int(row.minutes_played),
        round(float(row.minutes_share) * 1000),
        round(float(row.round_percentile_all) * 1000),
        round(float(row.peer_residual_z) * 100),
        round(float(row.ewma_z) * 100),
        round(float(row.cusum_positive) * 100),
    ])

def number(value, digits=2):
    return None if pd.isna(value) else round(float(value), digits)

def text_value(value):
    return None if pd.isna(value) else str(value)

def date_value(value):
    return None if pd.isna(value) else value.strftime('%Y-%m-%d')

players_payload = []
for row in summary.itertuples(index=False):
    players_payload.append({
        'id': str(row.player_id), 'name': row.player_name or str(row.player_id),
        'position': row.player_position or 'Unknown',
        'league': str(row.competition_id), 'club': str(row.club_name),
        'latestAge': number(row.latest_age), 'firstAge': number(row.first_age),
        'rounds': int(row.rounds), 'playedRounds': int(row.played_rounds),
        'minutes': int(row.total_minutes), 'playRate': number(row.play_rate, 3),
        'peakEwma': number(row.peak_ewma_z), 'latestEwma': number(row.latest_ewma_z),
        'maxCusum': number(row.max_cusum), 'score': number(row.signal_percentile, 1),
        'signalAge': number(row.signal_age),
        'signalDate': date_value(row.signal_date),
        'signalLeague': text_value(row.signal_competition_id),
        'signalClub': text_value(row.signal_club_name),
        'spells': int(row.league_spells), 'leagueCount': int(row.leagues),
        'ageAtCutoff': number(row.age_at_cutoff),
        'postSignalRounds': int(row.post_signal_rounds),
        'unusedStreak': int(row.unused_streak),
        'recent5Minutes': int(row.recent_5_minutes),
        'lastUsedDate': date_value(row.last_used_date),
        'lastObservedDate': date_value(row.last_observation_date),
        'nonUseFlag': bool(row.non_use_flag),
        'historicalUsageRounds': int(row.historical_usage_rounds),
        'recentUsageRounds': int(row.recent_usage_rounds),
        'historicalUsage': number(row.historical_usage_share, 4),
        'recentUsage': number(row.recent_usage_share, 4),
        'usageChangePp': number(row.usage_change_pp, 1),
        'usagePlotEligible': bool(row.usage_plot_eligible),
        'history': histories[str(row.player_id)],
    })

payload = {
    'meta': {
        **audit, 'minAge': MIN_AGE, 'maxAge': MAX_AGE,
        'ewmaAlpha': EWMA_ALPHA, 'cusumAllowance': CUSUM_ALLOWANCE,
        'minRoundsForRanking': MIN_ROUNDS_FOR_RANKING,
        'nonUseMinStreak': NON_USE_MIN_STREAK,
        'nonUseMinPostSignalRounds': NON_USE_MIN_POST_SIGNAL_ROUNDS,
        'usageHistoricalMinRounds': USAGE_HISTORICAL_MIN_ROUNDS,
        'usageRecentRounds': USAGE_RECENT_ROUNDS,
        'usageRecencyDays': USAGE_RECENCY_DAYS,
        'dataCutoff': data_cutoff.strftime('%Y-%m-%d'),
        'generatedAt': pd.Timestamp.now(tz='UTC').isoformat(),
        'signalLabel': 'descriptive opportunity signal; not a talent probability',
    },
    'leagues': [{'id': code, 'name': LEAGUE_NAMES[code]} for code in league_codes],
    'clubs': club_names,
    'historySchema': ['date', 'age_x1000', 'league_index', 'club_index', 'minutes',
                      'share_x1000', 'round_percentile_x1000', 'peer_z_x100',
                      'ewma_z_x100', 'cusum_x100'],
    'players': players_payload,
}
asset_path = ASSETS / 'player_opportunity_explorer.json'
asset_path.write_text(json.dumps(payload, ensure_ascii=False, separators=(',', ':'), allow_nan=False),
                      encoding='utf-8')
print(f'{asset_path}: {asset_path.stat().st_size / 1_000_000:.1f} MB')

/Users/gabrielcarbinatto/Documents/GitHub/statistical-player-opportunity-monitoring/assets/player_opportunity_explorer.json: 51.3 MB


In [6]:
display(summary.loc[summary['rank_eligible'], [
    'player_name', 'player_position', 'signal_age', 'signal_league_name',
    'signal_club_name', 'rounds', 'total_minutes', 'peak_ewma_z',
    'max_cusum', 'signal_percentile',
]].head(25))

,player_name,player_position,signal_age,signal_league_name,signal_club_name,rounds,total_minutes,peak_ewma_z,max_cusum,signal_percentile
0,Aaron Hickey,Defender,18.393259,Serie A,Bologna FC 1909,126,6934,4.000000,114.389676,99.996878
1,Andrea Papetti,Defender,18.007214,Serie A,Brescia Calcio (- 2025),12,917,4.000000,38.633815,99.996878
2,Rayan,Attack,19.30772,Brasileirão,Clube de Regatas Vasco da Gama,130,5603,3.995393,142.034945,99.987512
3,Tino Livramento,Defender,19.121543,Premier League,Southampton FC,131,7739,3.977163,136.597944,99.981269
4,Sulaiman Hazazi,Defender,18.116731,Saudi Pro League,Al-Wehda FC,115,3741,3.909992,62.029942,99.975025
5,Mateus Mané,Attack,18.450755,Premier League,Wolverhampton Wanderers,37,1791,3.883280,75.875607,99.968781
6,Roméo Lavia,Midfield,19.348789,Premier League,Southampton FC,100,3723,3.769444,82.138859,99.962537
7,Jair Cunha,Defender,20.244084,Brasileirão,Botafogo de Futebol e Regatas,64,1896,3.718526,41.567984,99.956294
8,Abdulmalik Al-Oyayari,Defender,20.055169,Saudi Pro League,Al-Taawoun FC,109,4322,3.708679,127.774134,99.950050
9,Wilfried Gnonto,Attack,19.288555,Premier League,Leeds United,69,2041,3.673639,57.093990,99.943806


## **6. Priority references**

- Wang, Berger & Burdick (2013), [Dynamic Item Response models](https://doi.org/10.1214/12-AOAS608).
- De Boeck et al. (2011), [IRT as an explanatory mixed model](https://doi.org/10.18637/jss.v039.i12).
- Proust-Lima et al. (2022), [longitudinal IRT in continuous time](https://doi.org/10.1016/j.ymeth.2022.01.005).
- Olsbjerg & Christensen (2015), [local dependence in longitudinal IRT](https://doi.org/10.3758/s13428-014-0553-0).
- Grigg & Spiegelhalter (2007), [risk-adjusted EWMA](https://doi.org/10.1198/016214506000001121).
- Steiner, Cook & Farewell (2000), [risk-adjusted CUSUM](https://doi.org/10.1093/biostatistics/1.4.441).
- Adams & MacKay (2007), [Bayesian online changepoint detection](https://arxiv.org/abs/0710.3742).
- Bergkamp et al. (2019), [methodological issues in talent identification](https://doi.org/10.1007/s40279-019-01113-w).
- Mills & Bullough (2026), [early trust and professional trajectories](https://doi.org/10.1177/17479541261416476).